# Open Insect ID v2 — Train the hierarchical Vision MoE

Set the accelerator to **GPU**. This notebook:
1. loads the curated manifest + images (attached as a Kaggle input dataset),
2. builds the taxonomy tree,
3. constructs the 4-level MoE, the hierarchical + focal + cost-matrix loss,
4. trains with optional Mixup/CutMix and BioCLIP distillation,
5. saves per-level checkpoints.


In [ ]:
import os, subprocess, sys

# --- Configure this to your repo -------------------------------------------
GITHUB_REPO = os.environ.get("OPENINSECT_REPO", "https://github.com/<you>/OpenInsectID.git")
# ---------------------------------------------------------------------------
IS_KAGGLE = os.path.exists("/kaggle/input")

if IS_KAGGLE and "OpenInsectID" not in os.getcwd():
    # Install the package straight from GitHub on a fresh Kaggle kernel.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", GITHUB_REPO])
else:
    # Local dev: ensure the repo root is importable.
    repo_root = os.getcwd()
    if not os.path.isdir(os.path.join(repo_root, "src")):
        repo_root = str(Path(GITHUB_REPO.split(".git")[0]).parent)
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

from pathlib import Path
import yaml

print("package ready; importable as `src.*`")


## Config

In [ ]:
import torch

from src.models import HierarchicalMoE, TaxonomyTree
from src.training import (
    Trainer, TrainConfig, CombinedTaxonomicLoss,
    train_transform, eval_transform, build_dataloaders,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
cfg = TrainConfig(
    epochs=30, lr=1e-3, weight_decay=1e-4,
    level_weights=(8.0, 4.0, 2.0, 1.0),   # order >> species
    focal_gamma=2.0, focal_alpha=0.25,    # long-tail on genus/species
    lambda_cost=1.0,                      # taxonomic cost-matrix strength
    mixup_alpha=0.4,
    device=device,
)
print("device:", device)


## Load manifest + build tree

In [ ]:
# On Kaggle, point these at your attached input dataset.
MANIFEST = "/kaggle/input/insectid-dataset/insectid_manifest.parquet"
IMAGE_ROOT = "/kaggle/input/insectid-dataset/images"

from pathlib import Path
if not Path(MANIFEST).exists():
    MANIFEST = "insectid_manifest.parquet"
    IMAGE_ROOT = "data/raw"

train_loader, val_loader, tree = build_dataloaders(
    MANIFEST, IMAGE_ROOT, batch_size=cfg.batch_size if hasattr(cfg, "batch_size") else 128,
    val_fraction=0.1, num_workers=2,
    transform=train_transform(image_size=224), seed=42,
)
print("orders:", len(tree.orders), "species:", tree.num_species)


## Build the MoE + loss

In [ ]:
model = HierarchicalMoE(
    tree, backbone_name="mnv4_small",
    tau_order=0.85, tau_levels=(0.85, 0.80, 0.70, 0.50),
)
n_params = sum(p.numel() for p in model.parameters())
print(f"MoE parameters: {n_params/1e6:.2f}M across "
      f"{len(model.family_experts)} family experts, "
      f"{len(model.genus_experts)} genus experts, "
      f"{len(model.species_experts)} species experts")

loss_fn = CombinedTaxonomicLoss(
    level_weights=cfg.level_weights, focal_gamma=cfg.focal_gamma,
    focal_alpha=cfg.focal_alpha, lambda_cost=cfg.lambda_cost,
)
loss_fn.build_cost_matrix(model)


## Distillation (optional)

Supply a BioCLIP teacher to transfer biologically-informed visual features.
Requires `pip install transformers` and a GPU. Leave `teacher = None` to skip.


In [ ]:
teacher = None  # set to True to attempt loading BioCLIP (needs transformers + GPU)
if teacher:
    from src.training.distillation import load_bioclip_teacher
    teacher = load_bioclip_teacher("imageomics/bioclip", device=device)
else:
    teacher = None
print("distillation:", "enabled" if teacher else "disabled")


## Train

In [ ]:
trainer = Trainer(model, cfg, teacher=teacher)
history = trainer.fit(train_loader, val_loader)


## Save checkpoints

In [ ]:
from pathlib import Path
out = Path("checkpoints"); out.mkdir(exist_ok=True)
torch.save(model.state_dict(), out / "moe_state.pt")
tree.save(out / "taxonomy.json")
print("saved:", list(p.name for p in out.iterdir()))

# On Kaggle, publish the checkpoint directory as an output dataset so the
# export notebook (03) can attach it:
#   !kaggle datasets create -p checkpoints --dir-mode tar
